# Demo: Model Overfitting

## Objectives

This notebook demonstrates **overfitting** - a common problem where a model fits the calibration data too well but fails to generalize to new observations. We explore:

- How increasing model complexity improves calibration fit
- Why a perfect calibration fit can be misleading
- The importance of validation data
- The bias-variance tradeoff

## Background

> 📚 **Definition: Overfitting**
> 
> Overfitting occurs when a model learns the noise in the calibration data rather than the underlying physical process. Signs of overfitting:
> - Excellent fit to calibration data
> - Poor fit to validation/new data
> - Physically unrealistic parameter values
> - Model complexity exceeds what data can support

In groundwater modeling, overfitting often happens when:
- Too many zones or pilot points are used
- Parameters vary spatially without physical justification
- The model "memorizes" observation locations rather than capturing flow patterns

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import polynomial as P
from IPython.display import display
import ipywidgets as widgets

# Set up plotting
%matplotlib widget
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 10

## 1. The Problem Setup

We simulate a simple scenario:
- **True system**: Head varies smoothly along a transect (quadratic function)
- **Observations**: Measured at a few locations with measurement noise
- **Models**: We fit polynomials of increasing complexity (1 to N parameters)

This is analogous to groundwater modeling where:
- Simple model = uniform K (few parameters)
- Complex model = many K zones (many parameters)

In [ ]:
def true_head_function(x):
    """
    The "true" underlying head distribution.
    A simple quadratic representing head decline with some curvature.
    """
    return 100 - 0.005 * x - 0.000003 * x**2


# Domain
x_domain = np.linspace(0, 1000, 200)
h_true = true_head_function(x_domain)

# Generate calibration observations (sparse, with noise)
np.random.seed(42)
n_cal = 6  # Number of calibration points
x_cal = np.array([50, 200, 400, 550, 750, 900])  # Calibration locations
noise_std = 0.3  # Measurement noise
h_cal = true_head_function(x_cal) + np.random.normal(0, noise_std, n_cal)

# Generate validation observations (different locations)
n_val = 5
x_val = np.array([120, 320, 500, 680, 850])  # Validation locations
h_val = true_head_function(x_val) + np.random.normal(0, noise_std, n_val)

print(f"Calibration points: {n_cal} observations")
print(f"Validation points: {n_val} observations")
print(f"Measurement noise: ±{noise_std} m (std)")

## 2. Fitting Models of Increasing Complexity

We fit polynomial models with 2 to 6 parameters (degrees 1 to 5):
- **2 parameters** (linear): h = a + bx
- **3 parameters** (quadratic): h = a + bx + cx²
- **6 parameters** (degree 5): h = a + bx + cx² + dx³ + ex⁴ + fx⁵

In [ ]:
def fit_and_evaluate(x_cal, h_cal, x_val, h_val, degree):
    """
    Fit a polynomial model and calculate calibration/validation errors.
    
    Args:
        x_cal, h_cal: Calibration data
        x_val, h_val: Validation data
        degree: Polynomial degree (n_parameters = degree + 1)
    
    Returns:
        coeffs: Fitted polynomial coefficients
        rmse_cal: Root mean square error on calibration data
        rmse_val: Root mean square error on validation data
    """
    # Fit polynomial to calibration data
    coeffs = np.polyfit(x_cal, h_cal, degree)
    
    # Evaluate on calibration points
    h_cal_pred = np.polyval(coeffs, x_cal)
    rmse_cal = np.sqrt(np.mean((h_cal_pred - h_cal)**2))
    
    # Evaluate on validation points
    h_val_pred = np.polyval(coeffs, x_val)
    rmse_val = np.sqrt(np.mean((h_val_pred - h_val)**2))
    
    return coeffs, rmse_cal, rmse_val


# Fit models with increasing complexity
degrees = [1, 2, 3, 4, 5]  # Polynomial degrees
results = {}

print("Model Complexity vs. Fit Quality:")
print("=" * 55)
print(f"{'Degree':^8} {'Parameters':^12} {'RMSE Cal':^12} {'RMSE Val':^12}")
print("-" * 55)

for deg in degrees:
    coeffs, rmse_cal, rmse_val = fit_and_evaluate(x_cal, h_cal, x_val, h_val, deg)
    results[deg] = {'coeffs': coeffs, 'rmse_cal': rmse_cal, 'rmse_val': rmse_val}
    n_params = deg + 1
    print(f"{deg:^8} {n_params:^12} {rmse_cal:^12.4f} {rmse_val:^12.4f}")

print("-" * 55)

## 3. Visualizing Overfitting

Let's see how different model complexities fit the data.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes = axes.flatten()

# Colors for different elements
colors = {'true': 'green', 'cal': 'blue', 'val': 'red', 'model': 'black'}

for idx, deg in enumerate(degrees):
    ax = axes[idx]
    
    # True function
    ax.plot(x_domain, h_true, '--', color=colors['true'], linewidth=1.5, alpha=0.7, label='True')
    
    # Calibration points
    ax.scatter(x_cal, h_cal, color=colors['cal'], s=60, zorder=5, label='Calibration', marker='o')
    
    # Validation points
    ax.scatter(x_val, h_val, color=colors['val'], s=60, zorder=5, label='Validation', marker='s')
    
    # Fitted model
    h_model = np.polyval(results[deg]['coeffs'], x_domain)
    ax.plot(x_domain, h_model, '-', color=colors['model'], linewidth=2, label='Model')
    
    # Labels
    rmse_c = results[deg]['rmse_cal']
    rmse_v = results[deg]['rmse_val']
    ax.set_title(f'{deg+1} parameters (degree {deg})\nCal: {rmse_c:.3f}, Val: {rmse_v:.3f}')
    ax.set_xlim(0, 1000)
    ax.set_ylim(91, 101)
    ax.grid(True, alpha=0.3)
    
    if idx == 0:
        ax.legend(loc='lower left', fontsize=8)
    if idx >= 3:
        ax.set_xlabel('Distance (m)')
    if idx % 3 == 0:
        ax.set_ylabel('Head (m)')

# Use last subplot for RMSE comparison
ax = axes[5]
n_params = [deg + 1 for deg in degrees]
rmse_cal_list = [results[deg]['rmse_cal'] for deg in degrees]
rmse_val_list = [results[deg]['rmse_val'] for deg in degrees]

ax.plot(n_params, rmse_cal_list, 'o-', color=colors['cal'], linewidth=2, markersize=8, label='Calibration RMSE')
ax.plot(n_params, rmse_val_list, 's-', color=colors['val'], linewidth=2, markersize=8, label='Validation RMSE')
ax.axhline(noise_std, color='gray', linestyle=':', label=f'Noise level ({noise_std})')
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('RMSE (m)')
ax.set_title('Calibration vs Validation Error')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(1.5, 6.5)

plt.tight_layout()
plt.show()

> 💡 **Key Observations**
> 
> - **Calibration RMSE** (blue) decreases as complexity increases - more parameters always fit calibration data better
> - **Validation RMSE** (red) initially decreases, then **increases** - the model starts fitting noise!
> - The **optimal model** is degree 2 (3 parameters) - matches the true underlying process
> - High-degree polynomials show wild oscillations between calibration points

## 4. Interactive Exploration

Use the slider to explore how model complexity affects the fit.

In [ ]:
fig_int, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: Model fit
ax1.plot(x_domain, h_true, '--', color='green', linewidth=1.5, alpha=0.7, label='True system')
ax1.scatter(x_cal, h_cal, color='blue', s=70, zorder=5, label='Calibration', marker='o')
ax1.scatter(x_val, h_val, color='red', s=70, zorder=5, label='Validation', marker='s')
model_line, = ax1.plot([], [], 'k-', linewidth=2, label='Model')
ax1.set_xlabel('Distance (m)')
ax1.set_ylabel('Head (m)')
ax1.set_title('Model Fit')
ax1.set_xlim(0, 1000)
ax1.set_ylim(91, 101)
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower left', fontsize=9)

info_text = ax1.text(0.98, 0.98, '', transform=ax1.transAxes, fontsize=10,
                     verticalalignment='top', horizontalalignment='right',
                     fontfamily='monospace',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

# Right: RMSE comparison
n_params = [deg + 1 for deg in degrees]
rmse_cal_list = [results[deg]['rmse_cal'] for deg in degrees]
rmse_val_list = [results[deg]['rmse_val'] for deg in degrees]

ax2.plot(n_params, rmse_cal_list, 'o-', color='blue', linewidth=2, markersize=8, label='Calibration')
ax2.plot(n_params, rmse_val_list, 's-', color='red', linewidth=2, markersize=8, label='Validation')
ax2.axhline(noise_std, color='gray', linestyle=':', alpha=0.7)
current_cal, = ax2.plot([], [], 'o', color='blue', markersize=15, markeredgecolor='black', markeredgewidth=2)
current_val, = ax2.plot([], [], 's', color='red', markersize=15, markeredgecolor='black', markeredgewidth=2)
ax2.set_xlabel('Number of Parameters')
ax2.set_ylabel('RMSE (m)')
ax2.set_title('Model Performance')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1.5, 6.5)
ax2.set_ylim(0, 1.2)

plt.tight_layout()

def update_plot(n_parameters):
    deg = n_parameters - 1
    
    # Update model line
    h_model = np.polyval(results[deg]['coeffs'], x_domain)
    model_line.set_data(x_domain, h_model)
    
    # Update info text
    rmse_c = results[deg]['rmse_cal']
    rmse_v = results[deg]['rmse_val']
    status = "UNDERFITTING" if deg < 2 else ("GOOD FIT" if deg == 2 else "OVERFITTING")
    info_text.set_text(f'{n_parameters} parameters\nCal RMSE: {rmse_c:.3f} m\nVal RMSE: {rmse_v:.3f} m\n{status}')
    
    # Update RMSE markers
    current_cal.set_data([n_parameters], [rmse_c])
    current_val.set_data([n_parameters], [rmse_v])
    
    fig_int.canvas.draw_idle()

# Create slider
param_slider = widgets.IntSlider(value=2, min=2, max=6, step=1,
                                  description='Parameters:',
                                  continuous_update=True,
                                  style={'description_width': 'initial'},
                                  layout=widgets.Layout(width='50%'))

widgets.interactive(update_plot, n_parameters=param_slider)
update_plot(2)

display(param_slider)
plt.show()

## 5. The Bias-Variance Tradeoff

> 📚 **Theory: Bias vs Variance**
> 
> Model error has two components:
> 
> - **Bias**: Error from oversimplifying the model (underfitting)
>   - Too few parameters to capture the true process
>   - Systematic errors in predictions
> 
> - **Variance**: Error from fitting noise (overfitting)  
>   - Too many parameters relative to data
>   - Predictions vary wildly with small data changes
> 
> The optimal model balances both - complex enough to capture physics, simple enough to avoid fitting noise.

| Model Complexity | Bias | Variance | Calibration Fit | Validation Fit |
|-----------------|------|----------|-----------------|----------------|
| Too simple | High | Low | Poor | Poor |
| **Optimal** | **Low** | **Low** | **Good** | **Good** |
| Too complex | Low | High | Excellent | Poor |

## 6. Implications for Groundwater Modeling

### Signs of Overfitting

- Calibration residuals much smaller than measurement uncertainty
- Large differences between calibration and validation performance
- Physically unrealistic parameter patterns (e.g., checkerboard K)
- Predictions very sensitive to small changes in observations

### How to Avoid Overfitting

1. **Use validation data**: Always reserve some observations for testing
2. **Regularization**: Penalize parameter complexity (e.g., Tikhonov in PEST)
3. **Parsimony**: Start simple, add complexity only when justified
4. **Cross-validation**: Test model on different data subsets
5. **Physical constraints**: Ensure parameters remain realistic

## 7. Key Takeaways

> 📚 **Summary: Overfitting**
> 
> 1. **More parameters ≠ better model** - complexity must match data support
> 
> 2. **Calibration fit can be misleading** - always check validation performance
> 
> 3. **Fitting noise reduces predictive power** - the model "memorizes" rather than "learns"
> 
> 4. **Optimal complexity** balances bias (underfitting) and variance (overfitting)
> 
> 5. **Regularization helps** - add penalties for unrealistic parameter variations

> ⚠️ **Practical Rule of Thumb**
> 
> If your calibration RMSE is much smaller than your measurement uncertainty, you're probably overfitting!

## References

- Hill, M.C. & Tiedeman, C.R. (2007). Effective Groundwater Model Calibration. Wiley.
- Doherty, J. & Hunt, R.J. (2010). Approaches to highly parameterized inversion. USGS SIR 2010-5169.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). The Elements of Statistical Learning. Springer.